## Importar librerías e instalar paquetes

In [1]:
import torch
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoFeatureExtractor

from torchvision import transforms
from tqdm import tqdm
import sklearn.metrics as metrics
import pandas as pd

## Usar GPU

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cpu


## Características de cada modelo

In [3]:
model_data = {
    "swinv2": {
        "image_size": 256,
        "model_name": "mirluvams/swinv2-base-patch4-window16-256-popocatepetl"
    },
    "beit": {
        "image_size": 224,
        "model_name": "mirluvams/beit-base-patch16-224-popocatepetl"
    },
    "vit": {
        "image_size": 224,
        "model_name": "mirluvams/vit-base-patch16-224-popocatepetl"
    },
    "convnext": {
        "image_size": 224,
        "model_name": "mirluvams//convnext-base-224-popocatepetl"
    },
    "dinov2": {
        "image_size": 224,
        "model_name": "mirluvams/dinov2-base-popocatepetl"
    }
}

In [4]:
model_data["swinv2"]["model_name"]

'mirluvams/swinv2-base-patch4-window16-256-popocatepetl'

## Cargar dataset de prueba

In [5]:
dataset = load_from_disk("../dataset")
test_dataset = dataset["test"]

In [6]:
print("Columnas:", test_dataset.column_names)
print("Cant. de imgs de test:", len(test_dataset))

Columnas: ['label', 'image', 'video', 'id']
Cant. de imgs de test: 24040


In [7]:
dataset["test"].set_format("torch")

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

## Función de evaluación de los modelos

In [19]:
BATCH_SIZE=128
def evaluate_model(model_name, image_size, **kwargs):
    print(f"\nEvaluando modelo: {model_name}")
    
    # Cargar modelo
    model = AutoModelForImageClassification.from_pretrained(model_name).to(DEVICE)
    
    # Transformar datos
    _test_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225]),
    ])
    
    def test_transform(ex):
        ex["pixel_values"] = [_test_transform(image) for image in ex["image"]]
        return ex
   
    dataset["test"].set_transform(test_transform)

    # Evaluar
    preds, labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(dataset["test"].iter(BATCH_SIZE), desc="Evaluando"):
            pixel_values = torch.stack(batch["pixel_values"]).to(DEVICE)
            batch_labels = batch["label"]
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch_labels)
    
    # Calcular métricas
    acc = metrics.accuracy_score(labels, preds)
    f1 = metrics.f1_score(labels, preds, average="weighted")
    f1_micro = metrics.f1_score(labels, preds, average="micro")
    f1_macro = metrics.f1_score(labels, preds, average="macro")
    
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | F1_micro: {f1_micro:.4f} | F1_macro: {f1_macro:.4f}")
    return {"model": model_name, "accuracy": acc, "f1": f1, "f1_micro": f1_micro, "f1_macro": f1_macro}

In [19]:
#evaluate_model(model_data["swinv2"]["model_name"], model_data["swinv2"]["image_size"])
evaluate_model(**model_data["swinv2"])


Evaluando modelo: mirluvams/swinv2-base-patch4-window16-256-popocatepetl


Evaluando: 2it [02:05, 62.77s/it]

KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
results = []

## Resultados de evaluación de los modelos

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results